# Global disease-level phenotype correlation analysis

**Data versions**

- Phenopacket Store: `v0.1.27`
- Human Phenotype Ontology: `v2026-06-23`

This notebook applies phenoco to analyze pairwise phenotype correlations
within disease-defined groups derived from the Phenopacket Store.

Phenopackets are grouped according to their observed disease annotations,
and each eligible disease group is analyzed independently. Pairwise
associations are quantified using the Phi coefficient, with statistical
significance assessed using Fisher's exact test. P-values are adjusted for
multiple testing using the Benjamini–Hochberg procedure within each disease
group.

Observed, excluded, and unreported HPO annotations are represented separately.
Ontologically related HPO term pairs are excluded from pairwise testing.
Analyses require a minimum disease-group size and a minimum effective sample
size for each phenotype pair.

The final sections summarize global results, perform quality-control checks,
and evaluate sensitivity to the effective-sample-size threshold.

#### 1. Imports and configuration

In [ ]:
from pathlib import Path
from contextlib import contextmanager
from zipfile import ZipFile
import os
import traceback

import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

from ppktstore.model import PhenopacketStore

from phenocor import (
    PhenotypeDatasetBuilder,
    HPOCorrelationAnalyzer,
)


# Data versions
PHENOPACKET_STORE_VERSION = "0.1.27"
HPO_RELEASE = "v2026-06-23"


# Project paths
PROJECT_ROOT = Path(
    os.environ.get(
        "phenocor_PROJECT_ROOT",
        "/Users/jingchen/Documents/ppkt2synergy/phenocor_case_studies",
    )
).expanduser().resolve()

STORE_ZIP = PROJECT_ROOT / "data" / "all_phenopackets.zip"
OUTDIR = PROJECT_ROOT / "results" / "global_analysis"
SENSITIVITY_DIR = OUTDIR / "sensitivity"


for directory in (
    OUTDIR,
    SENSITIVITY_DIR,

):
    directory.mkdir(parents=True, exist_ok=True)


# Analysis thresholds
MIN_DISEASE_SIZE = 20
MIN_PAIR_N = 20
FDR_THRESHOLD = 0.05


if not STORE_ZIP.is_file():
    raise FileNotFoundError(
        f"Phenopacket Store ZIP not found: {STORE_ZIP}"
    )

#### 2. Helper functions

In [2]:
def safe_fraction(numerator, denominator):
    if denominator == 0:
        return np.nan
    return numerator / denominator


def summarize_hpo_matrix(dataset):
    """
    Generate a compact summary of the phenotype matrix.
    """
    matrix = dataset.phenotype_matrix

    return {
        "n_hpo_terms": matrix.shape[1],
        "median_observed_terms_per_individual": float(
            (matrix == 1).sum(axis=1).median()
        ),
        "median_excluded_terms_per_individual": float(
            (matrix == 0).sum(axis=1).median()
        ),
        "median_explicit_terms_per_individual": float(
            matrix.notna().sum(axis=1).median()
        ),
    }

#### 3. Global scan

In [3]:
@contextmanager
def open_local_phenopacket_store(zip_path: Path):
    with ZipFile(zip_path, mode="r") as zip_file:
        store = PhenopacketStore.from_release_zip(
            zip_file=zip_file,
            strategy="lazy",
        )
        yield store

In [4]:
def group_phenopackets_by_disease(phenopackets):
    """
    Group Phenopackets by observed disease.

    Phenopackets annotated with multiple observed diseases are included
    in each corresponding disease group.

    Parameters
    ----------
    phenopackets : iterable
        Phenopackets from a single Phenopacket Store cohort.

    Returns
    -------
    dict[str, dict]
        Mapping from disease ID to disease name and Phenopackets.
    """
    disease_groups = {}

    for phenopacket in phenopackets:
        diseases = {}

        for disease in getattr(phenopacket, "diseases", []):
            if bool(getattr(disease, "excluded", False)):
                continue

            term = getattr(disease, "term", None)

            disease_id = getattr(term, "id", None)
            disease_name = getattr(term, "label", None)

            if not disease_id:
                continue

            disease_id = str(disease_id)
            disease_name = (
                str(disease_name)
                if disease_name
                else None
            )

            # One Phenopacket contributes only once to each disease.
            if disease_id not in diseases:
                diseases[disease_id] = disease_name

            # Prefer a non-empty label if another annotation provides one.
            elif (
                diseases[disease_id] is None
                and disease_name is not None
            ):
                diseases[disease_id] = disease_name

        for disease_id, disease_name in diseases.items():
            if disease_id not in disease_groups:
                disease_groups[disease_id] = {
                    "disease_name": disease_name,
                    "phenopackets": [],
                }

            elif (
                disease_groups[disease_id]["disease_name"] is None
                and disease_name is not None
            ):
                disease_groups[disease_id][
                    "disease_name"
                ] = disease_name

            disease_groups[disease_id][
                "phenopackets"
            ].append(phenopacket)

    return disease_groups

In [6]:
def run_global_disease_scan():
    """
    Run pairwise phenotype correlation analysis within disease-defined
    groups of Phenopacket Store cohorts.
    """
    status_rows = []
    disease_summary_rows = []
    tested_pair_tables = []
    significant_pair_tables = []
    error_rows = []

    with open_local_phenopacket_store(STORE_ZIP) as ps:

        cohort_names = sorted(
            cohort.name for cohort in ps.cohorts()
        )

        print(f"Total Store cohorts: {len(cohort_names)}")

        for position, cohort_name in enumerate(
            cohort_names,
            start=1,
        ):
            print(
                f"[{position}/{len(cohort_names)}] "
                f"Processing: {cohort_name}"
            )

            phenopackets = list(
                ps.iter_cohort_phenopackets(cohort_name)
            )

            disease_groups = group_phenopackets_by_disease(
                phenopackets
            )

            for disease_id, disease_group in sorted(
                disease_groups.items()
            ):
                disease_name = disease_group["disease_name"]
                disease_phenopackets = disease_group["phenopackets"]

                disease_group_size = len(disease_phenopackets)

                status_row = {
                    "cohort": cohort_name,
                    "disease_id": disease_id,
                    "disease_name": disease_name,
                    "disease_group_size": disease_group_size,
                    "status": None,
                    "reason": None,
                }

                # Disease-group size eligibility
                if disease_group_size < MIN_DISEASE_SIZE:
                    status_row["status"] = "excluded"
                    status_row["reason"] = (
                        f"fewer_than_{MIN_DISEASE_SIZE}_individuals"
                    )

                    status_rows.append(status_row)
                    continue

                try:
                    # Build disease-specific phenotype dataset
                    dataset = PhenotypeDatasetBuilder(
                        disease_phenopackets,
                        hpo_release=HPO_RELEASE,
                    ).build()

                    matrix_summary = summarize_hpo_matrix(
                        dataset
                    )

                    # Run pairwise correlation analysis
                    analyzer = HPOCorrelationAnalyzer(
                        dataset
                    )

                    correlation_result = (
                        analyzer.compute_correlation_matrix(
                            min_individuals=MIN_PAIR_N,
                            n_jobs=-1,
                        )
                    )

                    results_table = (
                        correlation_result
                        .correlation_results
                        .copy()
                    )

                    # No testable phenotype pairs
                    if results_table.empty:
                        status_row["status"] = (
                            "eligible_no_testable_pairs"
                        )
                        status_row["reason"] = (
                            "no_testable_pairs"
                        )

                        status_rows.append(status_row)

                        disease_summary_rows.append({
                            "cohort": cohort_name,
                            "disease_id": disease_id,
                            "disease_name": disease_name,
                            "disease_group_size": disease_group_size,
                            **matrix_summary,
                            "tested_pairs": 0,
                            "significant_pairs": 0,
                            "positive_significant_pairs": 0,
                            "negative_significant_pairs": 0,
                        })

                        continue

                    # Add disease-group metadata.
                    #
                    # Note:
                    # results_table already contains "n_individuals",
                    # which is the pair-specific effective sample size.
                    results_table.insert(
                        0,
                        "cohort",
                        cohort_name,
                    )
                    results_table.insert(
                        1,
                        "disease_id",
                        disease_id,
                    )
                    results_table.insert(
                        2,
                        "disease_name",
                        disease_name,
                    )
                    results_table.insert(
                        3,
                        "disease_group_size",
                        disease_group_size,
                    )

                    # Significant correlations
                    significant = (
                        correlation_result
                        .get_significant_correlations(
                            adj_pval_threshold=FDR_THRESHOLD,
                        )
                        .copy()
                    )

                    if not significant.empty:
                        significant.insert(
                            0,
                            "cohort",
                            cohort_name,
                        )
                        significant.insert(
                            1,
                            "disease_id",
                            disease_id,
                        )
                        significant.insert(
                            2,
                            "disease_name",
                            disease_name,
                        )
                        significant.insert(
                            3,
                            "disease_group_size",
                            disease_group_size,
                        )

                    n_positive = int(
                        (
                            significant["correlation"] > 0
                        ).sum()
                    )

                    n_negative = int(
                        (
                            significant["correlation"] < 0
                        ).sum()
                    )

                    tested_pair_tables.append(
                        results_table
                    )

                    if not significant.empty:
                        significant_pair_tables.append(
                            significant
                        )

                    disease_summary_rows.append({
                        "cohort": cohort_name,
                        "disease_id": disease_id,
                        "disease_name": disease_name,
                        "disease_group_size": disease_group_size,
                        **matrix_summary,
                        "tested_pairs": len(results_table),
                        "significant_pairs": len(significant),
                        "positive_significant_pairs": n_positive,
                        "negative_significant_pairs": n_negative,
                    })

                    if significant.empty:
                        status_row["status"] = (
                            "eligible_without_significant_results"
                        )
                    else:
                        status_row["status"] = (
                            "eligible_with_significant_results"
                        )

                    status_rows.append(status_row)

                except Exception as error:
                    status_row["status"] = "failed"
                    status_row["reason"] = type(error).__name__

                    status_rows.append(status_row)

                    error_rows.append({
                        "cohort": cohort_name,
                        "disease_id": disease_id,
                        "disease_name": disease_name,
                        "analysis": "disease_correlation",
                        "error_type": type(error).__name__,
                        "error_message": str(error),
                        "traceback": traceback.format_exc(),
                    })

    # ========================================================
    # Combine results
    # ========================================================

    status_df = pd.DataFrame(
        status_rows,
        columns=[
            "cohort",
            "disease_id",
            "disease_name",
            "disease_group_size",
            "status",
            "reason",
        ],
    )

    disease_summary_df = pd.DataFrame(
        disease_summary_rows,
        columns=[
            "cohort",
            "disease_id",
            "disease_name",
            "disease_group_size",
            "n_hpo_terms",
            "median_observed_terms_per_individual",
            "median_excluded_terms_per_individual",
            "median_explicit_terms_per_individual",
            "tested_pairs",
            "significant_pairs",
            "positive_significant_pairs",
            "negative_significant_pairs",
        ],
    )

    all_tested_df = (
        pd.concat(
            tested_pair_tables,
            ignore_index=True,
        )
        if tested_pair_tables
        else pd.DataFrame()
    )

    all_significant_df = (
        pd.concat(
            significant_pair_tables,
            ignore_index=True,
        )
        if significant_pair_tables
        else pd.DataFrame()
    )

    errors_df = pd.DataFrame(
        error_rows,
        columns=[
            "cohort",
            "disease_id",
            "disease_name",
            "analysis",
            "error_type",
            "error_message",
            "traceback",
        ],
    )

    # ========================================================
    # Internal consistency checks
    # ========================================================

    assert (
        int(disease_summary_df["tested_pairs"].sum())
        == len(all_tested_df)
    )

    assert (
        int(disease_summary_df["significant_pairs"].sum())
        == len(all_significant_df)
    )

    # ========================================================
    # Save results
    # ========================================================

    disease_summary_df.to_csv(
        OUTDIR / "disease_correlation_summary.csv",
        index=False,
    )

    all_significant_df.to_csv(
        OUTDIR / "significant_correlations.csv",
        index=False,
    )

    return {
        "status": status_df,
        "disease_summary": disease_summary_df,
        "all_tested_pairs": all_tested_df,
        "significant_pairs": all_significant_df,
        "errors": errors_df,
    }

#### 4. Run

In [7]:
disease_outputs = run_global_disease_scan()

status_df = disease_outputs["status"]
disease_summary_df = disease_outputs["disease_summary"]
all_tested_df = disease_outputs["all_tested_pairs"]
all_significant_df = disease_outputs["significant_pairs"]
errors_df = disease_outputs["errors"]

Total Store cohorts: 700
[1/700] Processing: 11q_terminal_deletion


Calculating pairwise correlations: 100%|██████████| 2016/2016 [00:01<00:00, 1550.42it/s]


[2/700] Processing: AAGAB
[3/700] Processing: ABCA4
[4/700] Processing: ABCB7
[5/700] Processing: ABCC8
[6/700] Processing: ACBD6


Calculating pairwise correlations: 100%|██████████| 2823/2823 [00:00<00:00, 23090.57it/s]


[7/700] Processing: ACP4
[8/700] Processing: ACTA1
[9/700] Processing: ACTB
[10/700] Processing: ACTN1
[11/700] Processing: ACVR1
[12/700] Processing: ADA
[13/700] Processing: ADA2
[14/700] Processing: ADAMTS10
[15/700] Processing: ADAMTS15
[16/700] Processing: ADAMTSL2
[17/700] Processing: ADGRG1
[18/700] Processing: ADRA2A
[19/700] Processing: AEBP1
[20/700] Processing: AFG3L2


Calculating pairwise correlations: 100%|██████████| 158/158 [00:00<00:00, 7446.15it/s]


... (203 outputs left out)

Calculating pairwise correlations: 100%|██████████| 1589/1589 [00:00<00:00, 18830.59it/s]


[695/700] Processing: ZNF462


Calculating pairwise correlations: 100%|██████████| 402/402 [00:00<00:00, 10181.33it/s]

[696/700] Processing: ZNF808
[697/700] Processing: ZNF862
[698/700] Processing: ZRSR2
[699/700] Processing: ZSCAN10
[700/700] Processing: ZSWIM6


#### 5. Results 

In [8]:
display(
    status_df["status"]
    .value_counts(dropna=False)
)

display(
    disease_summary_df
    .sort_values(
        "significant_pairs",
        ascending=False,
    )
    .head(20)
)
eligible_groups = disease_summary_df[
    disease_summary_df["disease_group_size"] >= MIN_DISEASE_SIZE
].copy()

eligible_groups = eligible_groups[
    eligible_groups["tested_pairs"] >= 1
].copy()

print(f"Eligible disease groups: {len(eligible_groups)}")
print(f"Mean group size: {eligible_groups['disease_group_size'].mean():.1f}")
print(f"Median group size: {eligible_groups['disease_group_size'].median():.1f}")
print(
    f"Range: {eligible_groups['disease_group_size'].min()}–"
    f"{eligible_groups['disease_group_size'].max()}"
)
display(errors_df)

status
excluded                                664
eligible_without_significant_results     66
eligible_with_significant_results        39
eligible_no_testable_pairs               12
Name: count, dtype: int64

,cohort,disease_id,disease_name,disease_group_size,n_hpo_terms,median_observed_terms_per_individual,median_excluded_terms_per_individual,median_explicit_terms_per_individual,tested_pairs,significant_pairs,positive_significant_pairs,negative_significant_pairs
77,RNU4-2,OMIM:620851,ReNU syndrome,61,210,25.0,139.0,166.0,11871,36,35,1
57,NF1,OMIM:162200,"Neurofibromatosis, type 1",405,36,3.0,17.0,20.0,412,27,26,1
21,FBXL4,OMIM:615471,Mitochondrial DNA depletion syndrome 13 (encep...,95,90,9.0,23.0,36.0,1117,20,20,0
35,INPP4A,OMIM:621354,Neurodevelopmental disorder with growth impair...,30,128,18.0,20.0,43.5,102,17,17,0
55,NBAS,OMIM:614800,"Short stature, optic nerve atrophy, and Pelger...",67,134,10.0,9.0,23.0,409,14,14,0
92,SON,OMIM:617140,ZTTK syndrome,64,354,23.0,70.0,96.0,2788,11,10,1
5,ANKRD11,OMIM:148050,KBG syndrome,333,48,7.0,6.0,12.0,895,9,9,0
17,EHMT1,OMIM:610253,Kleefstra syndrome 1,125,61,9.0,44.0,54.0,1588,7,6,1
45,LMNA,OMIM:151660,"Lipodystrophy, familial partial, type 2",127,60,9.0,8.0,16.0,91,6,5,1
65,POGZ,OMIM:616364,White-Sutton syndrome,117,94,8.0,2.0,10.0,109,6,6,0


Eligible disease groups: 105
Mean group size: 54.5
Median group size: 35.0
Range: 20–405


,cohort,disease_id,disease_name,analysis,error_type,error_message,traceback


In [9]:
# ============================================================
# Global disease-level results summary
# ============================================================

n_total_groups = len(status_df)

n_excluded_groups = int(
    (status_df["status"] == "excluded").sum()
)

n_eligible_groups = int(
    status_df["status"].str.startswith("eligible").sum()
)

n_no_testable_pairs = int(
    (
        status_df["status"]
        == "eligible_no_testable_pairs"
    ).sum()
)

n_groups_with_tested_pairs = int(
    (disease_summary_df["tested_pairs"] > 0).sum()
)

n_groups_with_significant_results = int(
    (disease_summary_df["significant_pairs"] > 0).sum()
)

n_tested_without_significant_results = int(
    (
        (disease_summary_df["tested_pairs"] > 0)
        & (disease_summary_df["significant_pairs"] == 0)
    ).sum()
)

n_tested_pairs = int(
    disease_summary_df["tested_pairs"].sum()
)

n_significant_pairs = int(
    disease_summary_df["significant_pairs"].sum()
)

n_positive_significant = int(
    disease_summary_df[
        "positive_significant_pairs"
    ].sum()
)

n_negative_significant = int(
    disease_summary_df[
        "negative_significant_pairs"
    ].sum()
)


global_summary_df = pd.DataFrame({
    "metric": [
        "Phenopacket Store version",
        "HPO release",
        "Minimum disease-group size",
        "Minimum pair-specific effective N",
        "FDR threshold",
        "Total disease groups",
        "Disease groups below size threshold",
        "Eligible disease groups",
        "Eligible groups without testable pairs",
        "Disease groups with tested pairs",
        "Disease groups with significant correlations",
        "Tested groups without significant correlations",
        "Total tested phenotype pairs",
        "Total significant phenotype pairs",
        "Positive significant correlations",
        "Negative significant correlations",
    ],
    "value": [
        PHENOPACKET_STORE_VERSION,
        HPO_RELEASE,
        MIN_DISEASE_SIZE,
        MIN_PAIR_N,
        FDR_THRESHOLD,
        n_total_groups,
        n_excluded_groups,
        n_eligible_groups,
        n_no_testable_pairs,
        n_groups_with_tested_pairs,
        n_groups_with_significant_results,
        n_tested_without_significant_results,
        n_tested_pairs,
        n_significant_pairs,
        n_positive_significant,
        n_negative_significant,
    ],
})

display(global_summary_df)

global_summary_df.to_csv(
    OUTDIR / "global_disease_correlation_summary.csv",
    index=False,
)

,metric,value
0,Phenopacket Store version,0.1.27
1,HPO release,v2026-06-23
2,Minimum disease-group size,20
3,Minimum pair-specific effective N,20
4,FDR threshold,0.05
5,Total disease groups,781
6,Disease groups below size threshold,664
7,Eligible disease groups,117
8,Eligible groups without testable pairs,12
9,Disease groups with tested pairs,105


In [10]:
display(
    disease_summary_df[
        [
            "cohort",
            "disease_id",
            "disease_name",
            "disease_group_size",
            "tested_pairs",
            "significant_pairs",
            "positive_significant_pairs",
            "negative_significant_pairs",
        ]
    ]
    .sort_values(
        ["significant_pairs", "disease_group_size"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

,cohort,disease_id,disease_name,disease_group_size,tested_pairs,significant_pairs,positive_significant_pairs,negative_significant_pairs
0,RNU4-2,OMIM:620851,ReNU syndrome,61,11871,36,35,1
1,NF1,OMIM:162200,"Neurofibromatosis, type 1",405,412,27,26,1
2,FBXL4,OMIM:615471,Mitochondrial DNA depletion syndrome 13 (encep...,95,1117,20,20,0
3,INPP4A,OMIM:621354,Neurodevelopmental disorder with growth impair...,30,102,17,17,0
4,NBAS,OMIM:614800,"Short stature, optic nerve atrophy, and Pelger...",67,409,14,14,0
...,...,...,...,...,...,...,...,...
112,SAMD9L,OMIM:159550,Ataxia-pancytopenia syndrome,21,0,0,0,0
113,SPTAN1,OMIM:620540,Developmental delay with or without epilepsy,21,103,0,0,0
114,MAPK8IP3,OMIM:618443,Neurodevelopmental disorder with or without va...,20,0,0,0,0
115,MAU2,OMIM:621570,Cornelia de Lange syndrome 7,20,1,0,0,0


# Effective-N sensitivity analysis

The primary analysis requires a pair-specific effective
sample size of at least 20 individuals.

Sensitivity analyses repeat the disease-level correlation
analysis using thresholds of 10, 20, and 30 individuals.
Disease-group eligibility is kept fixed at the primary
threshold (N >= 20). BH-FDR correction is recalculated
within each disease group at every effective-N threshold.

#### 1. Configuration and helpers

In [58]:
SENSITIVITY_THRESHOLDS = (10, 20, 30)
PRIMARY_PAIR_N = 20

SENSITIVITY_TABLE_PATH = (
    SENSITIVITY_DIR
    / "effective_n_sensitivity.csv"
)

In [60]:
def add_canonical_pair_columns(frame):
    """Add orientation-independent HPO-pair identifiers."""
    frame = frame.copy()

    ordered = np.sort(
        frame[
            ["HPO_A", "HPO_B"]
        ]
        .astype(str)
        .to_numpy(),
        axis=1,
    )

    frame["pair_hpo_1"] = ordered[:, 0]
    frame["pair_hpo_2"] = ordered[:, 1]

    return frame

def recalculate_fdr(frame):
    """Recalculate BH-FDR within one disease group."""
    frame = frame.loc[
        frame["p_value"].notna()
    ].copy()

    if frame.empty:
        frame["adj_p_value"] = pd.Series(dtype=float)
        frame["is_significant"] = pd.Series(dtype=bool)
        return frame

    _, adjusted, _, _ = multipletests(
        frame["p_value"].to_numpy(),
        method="fdr_bh",
    )

    frame["adj_p_value"] = adjusted
    frame["is_significant"] = (
        frame["adj_p_value"] < FDR_THRESHOLD
    )

    return frame

def pair_key_set(frame, key_columns):
    """Return unique analysis-unit and HPO-pair keys."""
    if frame.empty:
        return set()

    return set(
        map(
            tuple,
            frame[key_columns].to_numpy(),
        )
    )

#### 2. Association sensitivity analysis

In [61]:
def run_disease_correlation_sensitivity():
    """
    Run disease-level correlation analysis at the lowest
    effective-N threshold and derive stricter thresholds by
    filtering and recalculating BH-FDR.

    Disease-group eligibility is fixed at N >= MIN_DISEASE_SIZE.
    """
    minimum_threshold = min(
        SENSITIVITY_THRESHOLDS
    )

    unit_rows = []
    threshold_pair_tables = []
    error_rows = []

    with open_local_phenopacket_store(STORE_ZIP) as ps:

        cohort_names = sorted(
            cohort.name
            for cohort in ps.cohorts()
        )

        for position, cohort_name in enumerate(
            cohort_names,
            start=1,
        ):
            print(
                f"[{position}/{len(cohort_names)}] "
                f"Sensitivity: {cohort_name}"
            )

            phenopackets = list(
                ps.iter_cohort_phenopackets(
                    cohort_name
                )
            )

            disease_groups = (
                group_phenopackets_by_disease(
                    phenopackets
                )
            )

            for disease_id, disease_group in sorted(
                disease_groups.items()
            ):
                disease_name = (
                    disease_group["disease_name"]
                )

                disease_phenopackets = (
                    disease_group["phenopackets"]
                )

                disease_group_size = len(
                    disease_phenopackets
                )

                # Keep disease-group eligibility identical
                # to the primary analysis.
                if disease_group_size < MIN_DISEASE_SIZE:
                    continue

                try:
                    dataset = (
                        PhenotypeDatasetBuilder(
                            disease_phenopackets,
                            hpo_release=HPO_RELEASE,
                        )
                        .build()
                    )

                    analyzer = HPOCorrelationAnalyzer(
                        dataset
                    )

                    base_result = (
                        analyzer
                        .compute_correlation_matrix(
                            min_individuals=minimum_threshold,
                            n_jobs=-1,
                        )
                        .correlation_results
                        .copy()
                    )

                    if not base_result.empty:
                        base_result = (
                            add_canonical_pair_columns(
                                base_result
                            )
                        )

                    for threshold in (
                        SENSITIVITY_THRESHOLDS
                    ):
                        if base_result.empty:
                            result = pd.DataFrame()

                        else:
                            result = base_result.loc[
                                base_result[
                                    "n_individuals"
                                ] >= threshold
                            ].copy()

                            result = recalculate_fdr(
                                result
                            )

                        if not result.empty:
                            result.insert(
                                0,
                                "effective_n_threshold",
                                threshold,
                            )

                            result.insert(
                                1,
                                "cohort",
                                cohort_name,
                            )

                            result.insert(
                                2,
                                "disease_id",
                                disease_id,
                            )

                            result.insert(
                                3,
                                "disease_name",
                                disease_name,
                            )

                            result.insert(
                                4,
                                "disease_group_size",
                                disease_group_size,
                            )

                            threshold_pair_tables.append(
                                result
                            )

                            significant = result.loc[
                                result["is_significant"]
                            ]

                        else:
                            significant = pd.DataFrame()

                        unit_rows.append({
                            "effective_n_threshold": threshold,
                            "cohort": cohort_name,
                            "disease_id": disease_id,
                            "disease_name": disease_name,
                            "disease_group_size": disease_group_size,
                            "tested_pairs": len(result),
                            "significant_pairs": len(significant),
                            "positive_significant_pairs": (
                                int(
                                    (
                                        significant[
                                            "correlation"
                                        ] > 0
                                    ).sum()
                                )
                                if not significant.empty
                                else 0
                            ),
                            "negative_significant_pairs": (
                                int(
                                    (
                                        significant[
                                            "correlation"
                                        ] < 0
                                    ).sum()
                                )
                                if not significant.empty
                                else 0
                            ),
                        })

                except Exception as error:
                    error_rows.append({
                        "cohort": cohort_name,
                        "disease_id": disease_id,
                        "disease_name": disease_name,
                        "error_type": type(error).__name__,
                        "error_message": str(error),
                        "traceback": traceback.format_exc(),
                    })

    unit_df = pd.DataFrame(unit_rows)

    pairs_df = (
        pd.concat(
            threshold_pair_tables,
            ignore_index=True,
        )
        if threshold_pair_tables
        else pd.DataFrame()
    )

    errors_df = pd.DataFrame(error_rows)

    return {
        "unit_summary": unit_df,
        "pair_results": pairs_df,
        "errors": errors_df,
    }

#### 3. Run synergy sensitivity analysis

In [62]:
sensitivity_results = (
    run_disease_correlation_sensitivity()
)

sensitivity_errors = (
    sensitivity_results["errors"]
)

if not sensitivity_errors.empty:
    display(
        sensitivity_errors[
            [
                "cohort",
                "disease_id",
                "disease_name",
                "error_type",
                "error_message",
            ]
        ]
    )

    raise RuntimeError(
        "Sensitivity analysis did not complete "
        "without errors."
    )

[1/700] Sensitivity: 11q_terminal_deletion


Calculating pairwise correlations: 100%|██████████| 2079/2079 [00:01<00:00, 1441.40it/s]


[2/700] Sensitivity: AAGAB
[3/700] Sensitivity: ABCA4
[4/700] Sensitivity: ABCB7
[5/700] Sensitivity: ABCC8
[6/700] Sensitivity: ACBD6


Calculating pairwise correlations: 100%|██████████| 3067/3067 [00:00<00:00, 22526.13it/s]


[7/700] Sensitivity: ACP4
[8/700] Sensitivity: ACTA1
[9/700] Sensitivity: ACTB
[10/700] Sensitivity: ACTN1
[11/700] Sensitivity: ACVR1
[12/700] Sensitivity: ADA
[13/700] Sensitivity: ADA2
[14/700] Sensitivity: ADAMTS10
[15/700] Sensitivity: ADAMTS15
[16/700] Sensitivity: ADAMTSL2
[17/700] Sensitivity: ADGRG1
[18/700] Sensitivity: ADRA2A
[19/700] Sensitivity: AEBP1
[20/700] Sensitivity: AFG3L2


Calculating pairwise correlations: 100%|██████████| 166/166 [00:00<00:00, 6764.15it/s]


... (203 outputs left out)

Calculating pairwise correlations: 100%|██████████| 1687/1687 [00:00<00:00, 20831.06it/s]


[695/700] Sensitivity: ZNF462


Calculating pairwise correlations: 100%|██████████| 406/406 [00:00<00:00, 10358.20it/s]

[696/700] Sensitivity: ZNF808
[697/700] Sensitivity: ZNF862
[698/700] Sensitivity: ZRSR2
[699/700] Sensitivity: ZSCAN10
[700/700] Sensitivity: ZSWIM6


#### 4. Summarize counts and retention

In [63]:
def summarize_disease_correlation_sensitivity(
    sensitivity_results,
    primary_threshold=PRIMARY_PAIR_N,
):
    """
    Summarize effective-N sensitivity and retention of
    primary significant phenotype pairs.
    """
    unit_df = sensitivity_results["unit_summary"]
    pairs_df = sensitivity_results["pair_results"]

    summary_df = (
        unit_df
        .groupby(
            "effective_n_threshold",
            as_index=False,
        )
        .agg(
            eligible_disease_groups=(
                "disease_id",
                "size",
            ),
            disease_groups_with_tested_pairs=(
                "tested_pairs",
                lambda x: int((x > 0).sum()),
            ),
            disease_groups_with_significant_pairs=(
                "significant_pairs",
                lambda x: int((x > 0).sum()),
            ),
            tested_pairs=(
                "tested_pairs",
                "sum",
            ),
            significant_pairs=(
                "significant_pairs",
                "sum",
            ),
            positive_significant_pairs=(
                "positive_significant_pairs",
                "sum",
            ),
            negative_significant_pairs=(
                "negative_significant_pairs",
                "sum",
            ),
        )
    )

    key_columns = [
        "cohort",
        "disease_id",
        "pair_hpo_1",
        "pair_hpo_2",
    ]

    primary = pairs_df.loc[
        pairs_df["effective_n_threshold"]
        == primary_threshold
    ]

    primary_significant = primary.loc[
        primary["is_significant"]
    ]

    primary_keys = pair_key_set(
        primary_significant,
        key_columns,
    )

    stability_rows = []

    for threshold in summary_df[
        "effective_n_threshold"
    ]:
        comparison = pairs_df.loc[
            pairs_df["effective_n_threshold"]
            == threshold
        ]

        comparison_significant = comparison.loc[
            comparison["is_significant"]
        ]

        comparison_keys = pair_key_set(
            comparison,
            key_columns,
        )

        comparison_significant_keys = pair_key_set(
            comparison_significant,
            key_columns,
        )

        primary_testable = (
            primary_keys
            & comparison_keys
        )

        retained = (
            primary_testable
            & comparison_significant_keys
        )

        stability_rows.append({
            "effective_n_threshold": threshold,

            "primary_significant_pairs": (
                len(primary_keys)
            ),

            "primary_significant_pairs_testable": (
                len(primary_testable)
            ),

            "primary_significant_pairs_not_testable": (
                len(primary_keys)
                - len(primary_testable)
            ),

            "retained_primary_significant_pairs": (
                len(retained)
            ),

            "retention_among_testable_primary_pairs": (
                len(retained)
                / len(primary_testable)
                if primary_testable
                else np.nan
            ),
        })

    stability_df = pd.DataFrame(
        stability_rows
    )

    return summary_df.merge(
        stability_df,
        on="effective_n_threshold",
        how="left",
    )

In [64]:
effective_n_sensitivity_summary = (
    summarize_disease_correlation_sensitivity(
        sensitivity_results
    )
)

display(
    effective_n_sensitivity_summary
)

effective_n_sensitivity_summary.to_csv(
    SENSITIVITY_TABLE_PATH,
    index=False,
)

,effective_n_threshold,eligible_disease_groups,disease_groups_with_tested_pairs,disease_groups_with_significant_pairs,tested_pairs,significant_pairs,positive_significant_pairs,negative_significant_pairs,primary_significant_pairs,primary_significant_pairs_testable,primary_significant_pairs_not_testable,retained_primary_significant_pairs,retention_among_testable_primary_pairs
0,10,117,116,38,100418,208,192,16,212,212,0,167,0.787736
1,20,117,105,39,66169,212,195,17,212,212,0,212,1.000000
2,30,117,59,31,47158,170,154,16,212,157,55,157,1.000000


In [66]:
# ============================================================
# Save primary analysis outputs
# ============================================================

# Manuscript-oriented summary tables
global_summary_df.to_csv(
    TABLE_DIR / "global_disease_correlation_summary.csv",
    index=False,
)

disease_summary_df.to_csv(
    TABLE_DIR / "disease_correlation_summary.csv",
    index=False,
)

all_significant_df.to_csv(
    TABLE_DIR / "significant_correlations.csv",
    index=False,
)


# Complete tested results
all_tested_df.to_csv(
    COMPLETE_RESULTS_DIR / "all_tested_correlations.csv",
    index=False,
)


# Effective-N sensitivity analysis
effective_n_sensitivity_summary.to_csv(
    SENSITIVITY_TABLE_PATH,
    index=False,
)


# QC
status_df.to_csv(
    QC_DIR / "disease_analysis_status.csv",
    index=False,
)

errors_df.to_csv(
    QC_DIR / "disease_analysis_errors.csv",
    index=False,
)


print("Saved analysis outputs:")
print(f"  {TABLE_DIR / 'global_disease_correlation_summary.csv'}")
print(f"  {TABLE_DIR / 'disease_correlation_summary.csv'}")
print(f"  {TABLE_DIR / 'significant_correlations.csv'}")
print(f"  {COMPLETE_RESULTS_DIR / 'all_tested_correlations.csv'}")
print(f"  {SENSITIVITY_TABLE_PATH}")
print(f"  {QC_DIR / 'disease_analysis_status.csv'}")
print(f"  {QC_DIR / 'disease_analysis_errors.csv'}")

Saved analysis outputs:
  /Users/jingchen/Documents/ppkt2synergy/phenoco_case_studies/results/global_analysis/manuscript/tables/global_disease_correlation_summary.csv
  /Users/jingchen/Documents/ppkt2synergy/phenoco_case_studies/results/global_analysis/manuscript/tables/disease_correlation_summary.csv
  /Users/jingchen/Documents/ppkt2synergy/phenoco_case_studies/results/global_analysis/manuscript/tables/significant_correlations.csv
  /Users/jingchen/Documents/ppkt2synergy/phenoco_case_studies/results/global_analysis/complete_results/all_tested_correlations.csv
  /Users/jingchen/Documents/ppkt2synergy/phenoco_case_studies/results/global_analysis/sensitivity/effective_n_sensitivity.csv
  /Users/jingchen/Documents/ppkt2synergy/phenoco_case_studies/results/global_analysis/qc/disease_analysis_status.csv
  /Users/jingchen/Documents/ppkt2synergy/phenoco_case_studies/results/global_analysis/qc/disease_analysis_errors.csv
